# Beta — Risk-Factor Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building the **Beta** style factor exactly as MSCI describes it in *The Barra US Equity Model (USE4) — Empirical Notes, September 2011* (Appendix A, p.51) and *USE4 Methodology Notes, August 2011* (Sections 2.3 and 3.1). It is **not** a research notebook. The deliverable is a clean, USE4-faithful Beta factor written to parquet, suitable for input to a multi-factor risk model — and as the **upstream input** to the NLB and Residual Volatility factors.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as deeply untrustworthy until audited. Follow the stages linearly. Each stage has:
- ✅ **PDF SPEC** — exact verbatim quote or paraphrase from the USE4 documentation
- 🧪 **VALIDATE** — sanity checks to run after each stage

**Inputs:** Daily prices from `SHARADAR_SEP.csv`. No fundamentals, no analyst estimates.

**Deliverable:** A parquet file `beta_use4.parquet` keyed by `(permaticker, signal_date)` containing the standardized Beta exposure for every stock in the coverage universe on every monthly signal date.

**Beta is the simplest of the three** because there is no cubing, no orthogonalization, and no winsorization step in the USE4 spec for Beta. The pipeline is: estimate the time-series regression beta, then standardize cross-sectionally. That's it.

**Beta is also the most important style factor.** USE4 Table 4.2 reports avg \|t\| of 3.96–4.02 (highest among style factors) and a factor stability coefficient of 0.96–0.97. It is also the only style factor with non-trivial correlation to the cap-weighted ESTU (0.77–0.90), because high-beta stocks are mechanically tilted toward the market direction.

---

### Critical reading

The USE4 PDFs specify Beta in **three places**:
- **Empirical Notes, Section 3.4 (p.16)** — qualitative description ("The Beta factor is typically the most important style factor. It captures market risk that cannot be explained by the Country factor.")
- **Empirical Notes, Appendix A (p.51)** — formal descriptor definition (the regression and window/half-life)
- **Methodology Notes, Section 2.3 (p.9)** — the cap-weighted-mean / equal-weighted-std standardization rule that applies to every style factor

All three are quoted verbatim in §1 below.

## §1. The USE4 Beta spec — verbatim quotes

### 1a. Empirical Notes, Appendix A (p.51) — formal descriptor definition

> **Beta**
> 
> *Definition:* `1.0 · BETA`
> 
> *BETA — Beta (β):* *"Computed as the slope coefficient in a time-series regression of excess stock return, r_t − r_ft, against the cap-weighted excess return of the estimation universe R_t,*
> 
> `r_t − r_ft = α + β·R_t + e_t`     (Eq. A1)
> 
> *The regression coefficients are estimated over the trailing 252 trading days of returns with a half-life of 63 trading days."*

### 1b. Empirical Notes, Section 3.4 (p.16) — qualitative description

> *"The Beta factor is typically the most important style factor. It captures market risk that cannot be explained by the Country factor. We compute Beta by time-series regression of excess stock returns against the cap-weighted estimation universe, as described in Appendix A. To better understand how Beta relates to the Country factor, consider a fully invested long-only portfolio that is tilted toward high-beta stocks. Intuitively, this portfolio has greater market risk than a portfolio with a beta of 1. This additional market risk is captured through positive exposure to the Beta factor."*

### 1c. Methodology Notes, Section 2.3 (p.9) — standardization rule

> *"Descriptors are standardized to have a mean of 0 and a standard deviation of 1. In other words, if d_nl_Raw is the raw value of stock n for descriptor l, then the standardized descriptor value is given by*
> 
> `d_nl = (d_nl_Raw − μ_l) / σ_l`     (Eq. 2.4)
> 
> *where μ_l is the cap-weighted mean of the descriptor (within the estimation universe), and σ_l is the equal-weighted standard deviation. We adopt the convention of standardizing using the cap-weighted mean so that a well-diversified cap-weighted portfolio has approximately zero exposure to all style factors. For the standard deviation, however, we use equal weights to prevent large-cap stocks from having an undue influence on the overall scale of the exposures."*

### 1d. Methodology Notes, Section 2.3 (p.9) — single-descriptor style factors

Beta has only one descriptor (BETA itself, weight 1.0). The methodology says the final step of constructing a style factor is to *re-standardize* the exposures. For a single-descriptor factor with weight 1.0, this is mathematically equivalent to one standardization of the raw descriptor.

### 1e. Methodology Notes, Section 3.1 (p.13) — Beta is the regressor in cross-sectional regression with sqrt(mcap) weights

> *"Factor returns in USE4 are estimated using weighted least-squares regression, assuming that the variance of specific returns is inversely proportional to the square root of total market capitalization. This regression-weighting scheme reflects the empirical observation that the idiosyncratic risk of a stock decreases as the market capitalization of the firm increases."*

### 1f. Empirical Notes, Section 3.1 (p.7) — Estimation Universe

> *"The USE4 estimation universe utilizes the MSCI USA Investable Markets Index (USA IMI), which aims to reflect the full breadth of investment opportunities within the US market by targeting 99 percent of the float-adjusted market capitalization."*

---

**That is all the PDFs say about Beta construction.** Notably, the USE4 spec does NOT explicitly call for winsorization of Beta (unlike NLB/NLS, which have an explicit winsorize step). The descriptor goes straight to standardization. Section 2.2 of the Methodology Notes describes a general outlier-treatment algorithm that applies to all descriptors — see §9 for default treatment.

## §2. End-to-end pipeline

```
┌────────────────────────────────────────────────────────────────────┐
│  UPSTREAM:  estu_build.ipynb   →  estu_monthly.parquet             │
│             daily_build.ipynb  →  daily_returns.parquet (+ maps)   │
├────────────────────────────────────────────────────────────────────┤
│  STAGE 0:  Setup, parameters                                       │
│  STAGE 1:  Load shared daily-returns artifact                      │
│  STAGE 2:  Load shared ESTU                                        │
│  STAGE 3:  _td_to_sig map (benchmark already in daily.mkt_ret)     │
│  STAGE 4:  Per-stock Beta — EW-OLS, 252d window, 63d half-life     │
│  STAGE 5:  Outlier trim (3σ: CW mean ± 3·EW std, ESTU bounds)      │
│  STAGE 6:  Standardize (CW mean = 0, EW std = 1)                   │
│  STAGE 7:  Save deliverable                                        │
│  STAGE 8:  Validate                                                │
└────────────────────────────────────────────────────────────────────┘
```

**PIT discipline:** for any signal_date `t`, only data dated ≤ `t` is used.

## §3. Output schema — what the worker delivers

**Raw descriptor column:** `beta`
**Standardized score column:** `beta_score`

**File:** `data/out/beta_use4.parquet`

**Compression:** zstd, statistics=True

**Schema:**

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | End-of-month rebalance date (last trading day of month) |
| `in_estu` | Bool | Whether this stock was in the estimation universe on this date |
| `mcap` | Float64 | Market cap on signal_date (used for cap-weighting) |
| `beta` | Float64 | **Raw beta** from the time-series regression (Stage 4 output) |
| `alpha` | Float64 | Intercept from the time-series regression (kept for audit) |
| `beta_score` | Float64 | **Final Beta exposure** — standardized cross-sectionally (the deliverable) |
| `n_obs` | UInt32 | Number of return observations used to estimate beta |
| `r2` | Float64 | R² of the time-series regression — useful for quality screening |

**Coverage rules:**
- Compute `beta` and `beta_score` for **every stock with valid beta** (`n_obs ≥ MIN_OBS`), not just ESTU members
- Standardization moments (mean, std) are computed using **ESTU members only**
- Non-ESTU stocks get the *same* standardization parameters applied so the final factor is comparable across the coverage universe

This matches USE4's distinction between *coverage universe* (everyone gets a forecast) and *estimation universe* (subset used to estimate model parameters).

## §4. STAGE 0 — Setup, imports, paths

Standard imports. Use polars, not pandas, throughout (project convention).

```python

# ── Parameters ────────────────────────────────────────────────────────────────
# (Factor-specific parameters are defined in the Stage 0 code cell below.)
```

```python
# ───── Parameters ─────────────────────────────────────────────────────────────
FACTOR_TYPE = "time_series"

# ✅ PDF SPEC — Beta window and half-life
BETA_WINDOW_DAYS  = 252   # trailing 252 trading days
BETA_HALF_LIFE_TD = 63    # 63-day exponential half-life

# SUGGESTION (NOT IN PDF)
MIN_OBS        = 63                  # ~3 months minimum for a stable regression
CALENDAR_START = date(1999, 1, 1)
```

## §5. STAGE 1 — Load the shared daily panel (excess returns + ESTU benchmark)

In the intended order you built the shared daily panel first (step `01.5_daily`),
so this stage is just a load: read `daily_returns.parquet` and you already have
excess returns and the ESTU cap-weighted benchmark. If instead you took the
suggested detour and are building Beta before the panel, build the return
plumbing **inline** from the rules below — you will extract it into `01.5_daily`
afterward. Either way the rules are the same:

✅ **PDF SPEC:** excess returns (`r_t − r_ft`) regressed on the cap-weighted
estimation-universe return.

**SUGGESTION (NOT IN PDF):**
- RF: Ken French daily `RF` (percent → decimal); see `data/README.md` §11
- Returns: `ln(closeadj_t / closeadj_{t−1}) − rf_t` from SEP; drop null returns
- Dual-class dedup: keep the highest-mcap row per `(permaticker, date)`
- Benchmark: per trading day, cap-weight member excess returns by **prior-day**
  market cap (`mcap_lag`), membership from the owning signal date (most recent
  month-end ≤ date)

🧪 **VALIDATE:**
- ~39M return rows, ~17,400 tickers, ~6,900 trading days (1998-12 →)
- Benchmark correlation vs a broad market proxy > 0.90; full-sample ann. vol
  ≈ 15–20%; GFC and COVID windows > 40%; missing-benchmark dates < 30

**Heads-up:** every later time-series factor (Momentum, Residual Volatility,
NLB, NLS) reads this same panel. If you built it inline here rather than in
`01.5_daily`, extract it into the shared panel before moving on — you do not
want to reimplement this plumbing per factor (see
`01.5_daily/daily_spec.ipynb`).

## §6. STAGE 2 — Load the shared ESTU (`estu_monthly.parquet`)

✅ **PDF SPEC:** USE4 ESTU = MSCI USA Investable Markets Index (USA IMI) — ~99% of
US float-adjusted market cap, liquidity- and stability-screened, ~2,500–3,000 names.

**SUGGESTION:** load the shared `data/out/estu_monthly.parquet` built by
`estu_build.ipynb` (spec: `01_estu/estu_spec.ipynb`): top ~3,000
domestic common stocks on NYSE/NASDAQ/NYSEMKT, ATVR liquidity screen
(add ≥ 20%, retain ≥ 10%), 3,000/3,500 hysteresis buffer. **Every factor must use
this same ESTU** — factor-specific universes would break the multi-factor
cross-sectional regression.

🧪 **VALIDATE:**
- 330 monthly signal dates (1999-01-29 →); ESTU size mean ≈ 2,500, range ≈ 1,800–3,050
- Mega-caps (AAPL, MSFT, GOOGL, JPM) present every month they were public

## §7. STAGE 3 — ESTU benchmark (already in the daily artifact)

✅ **PDF SPEC:** factor regressions use the cap-weighted estimation-universe return.

**SUGGESTION:** `daily.mkt_ret` already carries the ESTU cap-weighted **excess**
benchmark, built and validated once in `daily_build.ipynb`. This stage only derives
`_td_to_sig` (trading day → owning signal date)
for the validation quintile checks.

🧪 **VALIDATE:** missing `mkt_ret` dates after the first signal date < 30.

## §8. STAGE 4 — Per-stock Beta

**The core computation.** This is your first time-series estimation — Momentum, Residual Volatility, and NLB all reuse the machinery you build here.

✅ **PDF SPEC:** OLS slope of `r_i,t − r_ft = α + β·R_t + e_t` over **252 trailing trading days** with **63-day exponential half-life weights**.

**Exponential weights:**

```
w_τ = exp(−ln(2) · (t − τ) / 63)
```

where `t` is the signal date and `τ` is the date of the observation. Most recent day weight = 1, day 63 days ago weight = 0.5, etc.

**Weighted OLS formula** (numerically stable form):

```
β = [Σ w·r·m − (Σ w·r)(Σ w·m)/Σw] / [Σ w·m² − (Σ w·m)²/Σw]
α = (Σ w·r − β · Σ w·m) / Σ w
```

where `r = ret`, `m = mkt_ret`, `w = exponential weight`.

**Compute for EVERY stock with positive mcap and sufficient observations**, not just ESTU members.

**PIT window logic:**
- For signal_date `sd`, window = the last 252 trading days `≤ sd`
- Use `bisect.bisect_right(sorted_trading_days, sd)` to find the index
- `window = sorted_trading_days[idx-252 : idx]`

**SUGGESTION:** `MIN_OBS = 63` (≈ 3 months). Stocks with fewer valid observations get `beta = null`.

🧪 **VALIDATE:**
- Cross-sectional median beta ≈ 1.0
- Cap-weighted mean beta on ESTU ≈ 1.0 (since the market is the cap-weighted ESTU portfolio by construction)
- Distribution unimodal, mostly in [0, 2.5] with some outliers in either tail
- AAPL/MSFT typically in [0.8, 1.4]; defensives (utilities, consumer staples) below 1.0; high-beta tech / small-caps above 1.5
- R² distribution: median ~0.3–0.5 for large-caps, lower for small/illiquid names
- Numerical equivalence check vs numpy reference, max diff < 1e-12

**Build this here from the formulas above.** Beta is your first time-series factor, so you write the estimator from scratch: a polars-vectorized `compute_betas_on_date()` for the production path, plus a small numpy-loop version to spot-check it. Keep both — every later time-series factor reuses this exact machinery.

## §9. STAGE 5 — Outlier treatment (optional but recommended)

**NOT IN PDF for Beta specifically.** The Empirical Notes Appendix A definition of BETA does **not** include a winsorization step. However, the Methodology Notes Section 2.2 (p.8) describes a general outlier-treatment algorithm that applies to all descriptors:

> *"We employ a multi-step algorithm to identify and treat outliers ... The first group represents values so extreme that they are treated as potential data errors and removed from the estimation process. The second group represents values that are regarded as legitimate, but nonetheless so large that their impact on the model must be limited. We trim these observations to three standard deviations from the mean."*

**SUGGESTION:** Apply 3-sigma trimming per signal_date, computed within ESTU. Beta values that are >3σ from the cap-weighted mean (within ESTU) get clipped to 3σ. This affects a small number of stocks (typically <1%) but prevents extreme estimation errors from poorly-traded names dominating the cross-section.

🧪 **VALIDATE:**
- ~0.5–2% of ESTU rows should be clipped at either bound per date
- The 3σ thresholds shift over time with cross-sectional dispersion
- After trimming, cross-sectional skewness and kurtosis of `beta` should be lower

## §10. STAGE 6 — Standardize  (cap-weighted mean=0, EW std=1)

✅ **PDF SPEC (Methodology Notes §2.3, Empirical Notes §3.4):** style factors are standardized to cap-weighted mean 0 and equal-weighted std 1.

**Per signal_date `t`, using only ESTU members:**

```
μ_t  = Σ_{i ∈ ESTU_t} (w_i · β_i)  /  Σ_{i ∈ ESTU_t} w_i        # cap-weighted mean
       where w_i = mcap_{i,t}

σ_t  = sqrt( mean_{i ∈ ESTU_t} [(β_i − μ_t)²] )                  # equal-weighted std

beta_score_{i,t} = (β_i − μ_t) / σ_t                              # applied to ALL i
```

Three critical points (same as NLB Stage 9):
1. **Moments are estimated on ESTU only.** Mean is cap-weighted; std is equal-weighted.
2. **The standardization is applied to every stock in the coverage universe.** Non-ESTU stocks get standardized using the ESTU's moments.
3. **Cap-weighting the mean using `mcap_{i,t}` directly** (not `sqrt(mcap)` — that's for joint cross-sectional regressions in factor return estimation, not for descriptor standardization).

**Why this convention?** Per Section 2.3 of the Methodology Notes: cap-weighting the mean ensures that a cap-weighted portfolio of ESTU has zero exposure to the Beta factor — i.e., the cap-weighted market portfolio is style-neutral. Equal-weighting the std prevents mega-caps from dominating the scale.

🧪 **VALIDATE:**
- `Σ_{i ∈ ESTU} (w_i · beta_score_i) / Σ w_i  ≈ 0`  (cap-weighted mean is 0 by construction)
- `std_{i ∈ ESTU} (beta_score_i) ≈ 1`  (equal-weighted std is 1 by construction)
- AAPL/MSFT/NVDA* (high-beta mega-caps) should have positive `beta_score` slightly above 0 (since cap-weighted mean tilts toward them)
- Utilities and consumer staples should have negative `beta_score` (low-beta defensives)
- Distribution: bell-shaped, mostly in [−3, +3] after standardization, range determined by underlying beta distribution

*These names will naturally change over time. No stock is garunteed to stay high-beta.

## §11. STAGE 7 — Save deliverable

Write the final panel to `data/out/beta_use4.parquet` with the schema from §3.

Compression: zstd. Statistics: True.

Include `beta` (raw), `alpha`, `r2`, `n_obs`, `mcap` for audit. The downstream consumer (NLB, Residual Volatility, multi-factor risk model) will use `beta_score`.

## §12. STAGE 8 — Validation

Run these checks **on the saved deliverable**.

### Required checks (all must pass before signing off)

**1. Standardization moments on ESTU:**
```
For each signal_date t, computed over ESTU:
    cap-weighted mean of beta_score ≈ 0   (|mean| < 1e-10)
    equal-weighted std of beta_score ≈ 1   (|std - 1| < 1e-6)
```

**2. Raw beta sanity check:**
```
For each signal_date t, computed over ESTU:
    cap-weighted mean of raw beta ≈ 1.0   (within ±0.05)
Reason: ESTU IS the cap-weighted market, so its beta against itself is 1 by construction.
```

**3. Coverage stability:**
```
Per signal_date, count of stocks with non-null beta_score should be stable over time.
Target: ≥ 4000 stocks per date in recent years.
```

**4. Factor stability (month-over-month rank correlation):**
```
For consecutive signal_dates t and t+1, compute Spearman correlation
of beta_score for stocks present in both dates.
USE4 reports Beta stability of 0.96-0.97 (Table 4.2). Target > 0.94.
Beta is moderately stable — half-life weighting introduces some noise, but the 252-day
window means month-to-month changes are gradual.
```

**5. Correlation with ESTU return series:**
```
Compute the long-short Q5-Q1 portfolio on beta_score, cap-weighted, monthly rebalance.
Daily return series of this portfolio should be highly correlated with daily market return.
USE4 Table 4.2 reports Beta-factor correlation with ESTU of 0.77-0.90. Target > 0.7.
```

**6. Equivalence vs numpy reference:**
```
Spot check at least one signal_date with your numpy-loop reference implementation.
Confirm max abs diff of beta values < 1e-12.
```

### Optional diagnostic checks

- Histogram of `beta_score` per signal_date — bell-shaped, centered ~0, range mostly [−3, +3]
- Time series of `cap-weighted mean(beta)` and `cap-weighted mean(beta_score)` — first should be ~1, second should be ~0
- Time series of cross-sectional std of raw beta — typically 0.4–0.6, spikes higher in crisis periods
- Top/bottom 20 by beta_score on a recent date — top should be small-cap biotech / fintech / leveraged ETFs; bottom should be utilities / staples / large-cap defensives

---

**Shared validation conventions (all factors, 2026-06-10):**
- **Coverage (check 3)** is evaluated on **completed months only** — the final
  signal date is the in-progress month (freshest data can lag the Sharadar
  refresh) and is excluded from the floor.
- **Fraction of scores in [−3, +3]** is reported for the full universe *and* for
  ESTU; the ESTU figure is the operationally relevant one (non-ESTU micro-caps
  pull the all-universe tail).
- Common check machinery lives in `common/`, your shared utilities.

## §13. Common pitfalls — read this before coding

**Pitfall 1: Mixing weighting schemes.**
- Standardization mean: **cap-weighted** (mcap)
- Standardization std: **equal-weighted** (no weights)
- Two different weighting schemes in two different places of the same standardization step. Don't homogenize them.

**Pitfall 2: Standardizing on the wrong universe.**
- Moments (mean, std) are computed using **ESTU only**
- The standardization is **applied to every stock** in the coverage universe
- This is the point of having a coverage universe vs an estimation universe

**Pitfall 3: Forgetting the intercept in the time-series regression.**
- USE4 Equation A1 explicitly includes the intercept `α`
- Omitting it (regressing through the origin) biases the slope estimate
- This is especially severe if returns have non-zero mean over the window (which they typically do in long bull/bear runs)

**Pitfall 4: PIT violations.**
- For signal_date `t`, the 252-day window must contain only dates `≤ t`
- Use `bisect.bisect_right(sorted_trading_days, t)` to find the right slice
- Get this slice logic right here — the same PIT window recurs in Momentum, Residual Volatility, and NLB

**Pitfall 5: Using `mcap` instead of `mcap_lag` for the daily market benchmark.**
- The market return for day `t` uses each stock's market cap **at end of day `t−1`**
- Using `t`'s market cap is look-ahead

**Pitfall 6: Beta of the market portfolio against itself.**
- The cap-weighted-ESTU portfolio's beta against itself is 1.0 by definition
- Cap-weighted mean of raw betas across ESTU should be ~1.0
- If your computation gives something materially different (e.g., 0.7 or 1.3), there's a bug — likely in the benchmark definition or in the regression

**Pitfall 7: Half-life weighting in the OLS regression.**
- USE4 spec is **exponentially weighted** OLS with 63-day half-life
- Standard unweighted OLS will give different (less responsive) betas
- The weighted formula is in Stage 4 — make sure you implement the weighted version

**Pitfall 8: NaN handling in the standardization step.**
- Stocks with NaN beta (insufficient obs) should NOT enter the cap-weighted mean or EW std
- Filter `drop_nulls(subset=["beta"])` *before* computing moments
- Similarly for mcap — `null` or `<= 0` market cap shouldn't get weight

**Pitfall 9: Confusing Beta the descriptor with Beta the factor return.**
- Beta the **descriptor** (this notebook) is the exposure value per stock
- Beta the **factor return** comes from running the cross-sectional WLS regression of stock returns on all factor exposures — that's a downstream step, NOT what this notebook produces
- This notebook delivers `beta_score`, the exposure. The factor return is computed later when you run the multi-factor model.

**Pitfall 10: Beta's correlation with the Country factor is a feature.**
- USE4 Table 4.2 reports Beta-ESTU correlations of 0.77-0.90
- This is expected and intended — high-beta stocks ARE mechanically tilted toward the market
- Don't try to "clean" this by orthogonalizing Beta to the Country factor — that breaks the USE4 spec and the interpretation of the factor
- The Country factor and Beta factor are designed to be additive risk sources when a portfolio is long high-beta stocks

## §14. Final summary — what "done" looks like

**The deliverable is one file:** `data/out/beta_use4.parquet`

**Acceptance criteria:**

1. ✅ Schema matches §3
2. ✅ All 6 required validation checks in §12 pass within tolerance
3. ✅ ESTU has ~2500–3000 names per date, stable over time (from your ESTU build, step 01)
4. ✅ Cap-weighted mean of beta_score is machine-zero for every date in ESTU
5. ✅ Equal-weighted std of beta_score is 1.0 (to 6 decimals) for every date in ESTU
6. ✅ Cap-weighted mean of RAW beta is ≈ 1.0 (within ±0.05) for every date in ESTU
7. ✅ Coverage of beta_score across coverage universe is ≥ 4000 stocks per date in recent years
8. ✅ Month-over-month rank stability ρ > 0.94 for ESTU members
9. ✅ Numerical equivalence between the polars build and a numpy reference, max abs diff < 1e-12

If all 9 are satisfied, Beta is risk-model-ready and serves as the foundational input for NLB and Residual Volatility.

---

**The reusable pieces** (the first three live in the shared daily panel, `01.5_daily` — if you took the detour of building Beta before the panel, you write them inline here and extract them into `01.5_daily` afterward; the last two are built here and reused by every later factor):
- the price loader and prices → excess log returns
- the cap-weighted (`mcap_lag`-weighted) ESTU market return `mkt_ret`
- monthly signal-calendar construction
- `compute_betas_on_date()`, polars-vectorized, plus a `_compute_betas_reference()` numpy version for spot-checks
- cap-weighted-mean / EW-std standardization

**Beta is the foundational style factor.** Get it right and the downstream factors (NLB, Residual Volatility) inherit its correctness. Get it wrong and every downstream factor is biased in the same direction.

**Pro tip:** The shared machinery between Beta, NLB, NLS, and Residual Volatility (sqrt-mcap WLS orthogonalization, cap-weighted-mean / EW-std standardization, exponentially-weighted OLS) is worth factoring into reusable helpers in a shared `common/` module once you have 2–3 of these factors built. The same patterns will repeat for every descriptor in the USE4 model.